In [ ]:
import numpy as np
from scipy import sparse
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.validation import check_is_fitted

TYPES_VALIDES = {"numerical": "Numerical", "categorical": "Categorical", "binary": "Binary"}


def _inf_vers_nan(X):
    """Remplace +/-inf par NaN sans toucher aux zéros implicites d'une matrice sparse."""
    if sparse.issparse(X):
        X = X.tocsr(copy=True)
        X.data = X.data.astype(float, copy=False)
        X.data[np.isinf(X.data)] = np.nan
        return X
    X = np.array(X, dtype=float, copy=True)
    X[np.isinf(X)] = np.nan
    return X


class PretraitementAuto(BaseEstimator, TransformerMixin):
    """Prétraitement piloté par les types du fichier .type (Numerical / Categorical / Binary).

    types          : liste de chaînes, une par colonne (dataset.types).
    strategie_cat  : "frequente" (NaN -> modalité la plus fréquente du train)
                     ou "manquant" (NaN -> modalité dédiée).
    mise_a_echelle : standardise les colonnes numériques ; sur sparse, sans centrage.
    sortie_creuse  : True / False, ou None = même format que l'entrée vue au fit.
    """

    def __init__(self, types, strategie_cat="frequente", mise_a_echelle=False, sortie_creuse=None):
        self.types = types
        self.strategie_cat = strategie_cat
        self.mise_a_echelle = mise_a_echelle
        self.sortie_creuse = sortie_creuse

    def _indices_par_type(self, n_colonnes):
        if len(self.types) != n_colonnes:
            raise ValueError(
                f"{len(self.types)} types déclarés pour {n_colonnes} colonnes : incohérence."
            )
        normalises = []
        for t in self.types:
            cle = str(t).strip().casefold()
            if cle not in TYPES_VALIDES:
                raise ValueError(f"Type de colonne inconnu : {t!r} (attendu : {sorted(TYPES_VALIDES.values())})")
            normalises.append(TYPES_VALIDES[cle])
        return {
            nom: [i for i, t in enumerate(normalises) if t == nom]
            for nom in TYPES_VALIDES.values()
        }

    def fit(self, X, y=None):
        if self.strategie_cat not in ("frequente", "manquant"):
            raise ValueError("strategie_cat doit valoir 'frequente' ou 'manquant'.")
        if X.ndim != 2:
            raise ValueError("X doit être une matrice 2D.")

        X = _inf_vers_nan(X)
        self.n_features_in_ = X.shape[1]
        self.entree_creuse_ = sparse.issparse(X)
        self.sortie_creuse_ = self.entree_creuse_ if self.sortie_creuse is None else bool(self.sortie_creuse)
        idx = self._indices_par_type(self.n_features_in_)

        etapes_num = [("imputer", SimpleImputer(strategy="median", keep_empty_features=True))] 
        if self.mise_a_echelle:
            etapes_num.append(("echelle", StandardScaler(with_mean=not self.entree_creuse_)))

        etapes_bin = [("imputer", SimpleImputer(strategy="most_frequent", keep_empty_features=True))] 

        if self.strategie_cat == "frequente":     #modalités inconnues ignorées au transform
            imp_cat = SimpleImputer(strategy="most_frequent", keep_empty_features=True)
        else:
            imp_cat = SimpleImputer(strategy="constant", fill_value=-1, keep_empty_features=True)
        etapes_cat = [
            ("imputer", imp_cat),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
        ]

        blocs = []
        for nom, etapes, cols in (
            ("num", etapes_num, idx["Numerical"]),
            ("bin", etapes_bin, idx["Binary"]),
            ("cat", etapes_cat, idx["Categorical"]),
        ):
            if cols:
                blocs.append((nom, Pipeline(etapes), cols))
        if not blocs:
            raise ValueError("Aucune colonne à traiter.")

        self.transformeur_ = ColumnTransformer(
            blocs, remainder="drop", sparse_threshold=1.0 if self.sortie_creuse_ else 0.0
        )
        self.transformeur_.fit(X)
        return self

    def transform(self, X):
        check_is_fitted(self, "transformeur_")
        if X.shape[1] != self.n_features_in_:
            raise ValueError(
                f"X a {X.shape[1]} colonnes, le prétraitement en a appris {self.n_features_in_}."
            )
        Xt = self.transformeur_.transform(_inf_vers_nan(X))
        if self.sortie_creuse_ and not sparse.issparse(Xt):
            Xt = sparse.csr_matrix(Xt)
        elif not self.sortie_creuse_ and sparse.issparse(Xt):
            Xt = Xt.toarray()  # choix explicite : seulement si la taille le permet
        return Xt

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, "transformeur_")
        return self.transformeur_.get_feature_names_out()